# Example analysis: Capstone Global Health Retail

This notebook demonstrates how to load the synthetic data generated by `data/generate_synthetic_data.py`, perform the necessary joins, and run common analyses useful for a capstone project: cohorting, top products, country comparisons, and time series trends.

Requirements: run the generator first to produce the CSVs in `data/output` (see `data/README.md`).


In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

sns.set(style="whitegrid")
%matplotlib inline


In [ ]:
# Paths to the generated CSVs - change if you used a different out_dir
base = 'data/output'
products_path = f'{base}/products.csv'
stores_path = f'{base}/stores.csv'
customers_path = f'{base}/customers.csv'
transactions_path = f'{base}/transactions.csv'
transactions_enriched_path = f'{base}/transactions_enriched.csv'

# Load CSVs (transactions_enriched already has joins but we'll show how to join from raw transactions too)
products = pd.read_csv(products_path)
stores = pd.read_csv(stores_path)
customers = pd.read_csv(customers_path)
transactions = pd.read_csv(transactions_path, parse_dates=['transaction_date'])

# If transactions_enriched exists, load it as well for convenience
try:
    enriched = pd.read_csv(transactions_enriched_path, parse_dates=['transaction_date', 'signup_date'])
except FileNotFoundError:
    enriched = None

print('products', products.shape)
print('stores', stores.shape)
print('customers', customers.shape)
print('transactions', transactions.shape)
print('enriched loaded', enriched is not None)


## Join transactions with customers, products, and stores (if not already enriched)

In [ ]:
if enriched is None:
    enriched = transactions.merge(customers, on='customer_id', how='left')
    enriched = enriched.merge(products, on='product_id', how='left')
    enriched = enriched.merge(stores, on='store_id', how='left')
else:
    # ensure date columns parsed
    enriched['transaction_date'] = pd.to_datetime(enriched['transaction_date'])
    if 'signup_date' in enriched.columns:
        enriched['signup_date'] = pd.to_datetime(enriched['signup_date'])

enriched.head()


## 1) Cohorting (by customer signup month) - simple retention table

In [ ]:
# Prepare cohort columns
enriched['signup_month'] = pd.to_datetime(enriched['signup_date']).dt.to_period('M')
enriched['transaction_month'] = enriched['transaction_date'].dt.to_period('M')

# cohort index = months since signup
enriched['cohort_index'] = (enriched['transaction_month'] - enriched['signup_month']).apply(lambda x: x.n)

# cohort size (unique customers by signup_month)
cohort_sizes = customers.copy()
cohort_sizes['signup_month'] = pd.to_datetime(cohort_sizes['signup_date']).dt.to_period('M')
cohort_sizes = cohort_sizes.groupby('signup_month').agg(n_customers=('customer_id','nunique')).reset_index()

# retention: for each signup_month and cohort_index, count unique customers with a transaction
retention = enriched.groupby(['signup_month','cohort_index']).agg(active_customers=('customer_id','nunique')).reset_index()
retention = retention.merge(cohort_sizes, on='signup_month')
retention['retention_rate'] = retention['active_customers'] / retention['n_customers']

# Pivot for heatmap
retention_pivot = retention.pivot(index='signup_month', columns='cohort_index', values='retention_rate').fillna(0)

plt.figure(figsize=(12,8))
sns.heatmap(retention_pivot, annot=True, fmt='.2f', cmap='Blues')
plt.title('Cohort Retention (signup month vs months since signup)')
plt.ylabel('Signup month')
plt.xlabel('Months since signup (cohort index)')
plt.show()


## 2) Top products by revenue and quantity

In [ ]:
top_by_revenue = enriched.groupby(['product_id','product_name']).agg(total_revenue=('total_amount','sum'), total_qty=('quantity','sum')).reset_index()
top_by_revenue = top_by_revenue.sort_values('total_revenue', ascending=False).head(20)

plt.figure(figsize=(10,6))
sns.barplot(data=top_by_revenue.head(10), y='product_name', x='total_revenue', palette='viridis')
plt.title('Top 10 Products by Revenue')
plt.xlabel('Total revenue')
plt.ylabel('Product')
plt.tight_layout()
plt.show()

top_by_qty = top_by_revenue.sort_values('total_qty', ascending=False).head(10)
top_by_qty[['product_name','total_qty']].head(10)


## 3) Country comparisons (store country and customer country)

In [ ]:
# Sales by store country
sales_by_store_country = enriched.groupby('country').agg(total_revenue=('total_amount','sum'), transactions=('transaction_id','nunique')).reset_index().sort_values('total_revenue', ascending=False)

plt.figure(figsize=(10,6))
sns.barplot(data=sales_by_store_country.head(10), x='total_revenue', y='country', palette='magma')
plt.title('Top 10 Store Countries by Revenue')
plt.xlabel('Total revenue')
plt.ylabel('Country')
plt.show()

# Compare customer country vs store country - fraction of cross-border matches
cross = enriched.copy()
cross['is_local'] = cross['country_x'] == cross['country_y'] if 'country_x' in cross.columns else (cross['country'] == cross['country'])
# The generator uses 'country' for stores; customers also have 'country' but column names may collide after merges.
# Let's make explicit columns if possible
if 'country_x' in enriched.columns and 'country_y' in enriched.columns:
    cross['store_country'] = enriched['country_x']
    cross['customer_country'] = enriched['country_y']
else:
    # attempt to infer by checking store_id presence in stores
    if 'store_id' in stores.columns and 'country' in stores.columns:
        stores_map = stores.set_index('store_id')['country'].to_dict()
        customers_map = customers.set_index('customer_id')['country'].to_dict()
        cross['store_country'] = cross['store_id'].map(stores_map)
        cross['customer_country'] = cross['customer_id'].map(customers_map)

# group by store country
sales_by_store_country = cross.groupby('store_country').agg(total_revenue=('total_amount','sum'), transactions=('transaction_id','nunique')).reset_index().sort_values('total_revenue', ascending=False)
sales_by_store_country.head(10)


## 4) Time series: daily / monthly sales and seasonality

In [ ]:
ts = enriched.set_index('transaction_date').sort_index()
daily = ts['total_amount'].resample('D').sum().fillna(0)
monthly = ts['total_amount'].resample('M').sum()

plt.figure(figsize=(14,4))
plt.plot(daily.index, daily.values, label='Daily sales')
plt.title('Daily Sales')
plt.xlabel('Date')
plt.ylabel('Total sales')
plt.legend()
plt.show()

plt.figure(figsize=(12,4))
plt.plot(monthly.index, monthly.values, marker='o')
plt.title('Monthly Sales')
plt.xlabel('Month')
plt.ylabel('Total sales')
plt.show()

# Decompose a short seasonal view by weekday
daily_df = daily.reset_index().rename(columns={'transaction_date':'date', 0:'sales'}) if isinstance(daily, pd.Series) else daily.reset_index().rename(columns={ 'total_amount':'sales'})
daily_df['weekday'] = daily_df['transaction_date'].dt.day_name() if 'transaction_date' in daily_df.columns else daily_df['date'].dt.day_name()
weekday_sales = daily_df.groupby('weekday').agg(mean_sales=('sales','mean')).reindex(['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday'])
plt.figure(figsize=(8,4))
sns.barplot(x=weekday_sales.index, y='mean_sales', data=weekday_sales.reset_index(), palette='coolwarm')
plt.xticks(rotation=45)
plt.title('Average Sales by Weekday')
plt.show()


## Wrap-up
This notebook demonstrated basic steps to load the generated CSVs, join them, and run some common analytical patterns for a retail dataset. You can extend it with:
- Customer lifetime value (CLV) calculations
- Funnel / conversion analyses (visits -> purchases) if event data exists
- More detailed cohort retention visualizations by product category or country
- Forecasting models (ARIMA, Prophet) for monthly sales
